In [ ]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        os.path.join(dirname, filename)


In [ ]:
from fastai.vision.all import get_image_files
import torch
import torchvision.transforms as transforms
import torchvision.datasets as datasets
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.resnet50 import preprocess_input
import seaborn as sns
import matplotlib.pyplot as plt
import shutil
from sklearn.model_selection import train_test_split

In [ ]:
path = "/kaggle/input/emotion-classifier-color"
dataset_path = os.path.join(path, 'dataset')
if os.path.exists(dataset_path) and os.path.isdir(dataset_path):
    contents = os.listdir(dataset_path)
print(contents)

In [ ]:
# Define paths
dataset_path = "/kaggle/input/emotion-classifier-color/dataset"
output_path = "/kaggle/working/emotion-dataset"
train_path = os.path.join(output_path, 'train')
val_path = os.path.join(output_path, 'val')

def clear_folder(folder_path):
    if os.path.exists(folder_path):
        for filename in os.listdir(folder_path):
            file_path = os.path.join(folder_path, filename)
            try:
                if os.path.isfile(file_path) or os.path.islink(file_path):
                    os.unlink(file_path)
                elif os.path.isdir(file_path):
                    shutil.rmtree(file_path)
            except Exception as e:
                print(f"Failed to delete {file_path}. Reason: {e}")

# Clear the 'train' and 'val' folders
clear_folder(train_path)
clear_folder(val_path)

# Create train/val directories if they don't already exist
os.makedirs(train_path, exist_ok=True)
os.makedirs(val_path, exist_ok=True)

# Excluded folders
excluded_folders = ['Disgust', 'Fear']

# Ensure the dataset exists and is a directory
if os.path.exists(dataset_path) and os.path.isdir(dataset_path):
    contents = os.listdir(dataset_path)
    print("Dataset contents:", contents)

    # Filter out the excluded folders
    for emotion in contents:
        if emotion in excluded_folders:  # Skip excluded folders
            print(f"Skipping folder: {emotion}")
            continue

        emotion_folder = os.path.join(dataset_path, emotion)

        # Check if the current item is a folder
        if not os.path.isdir(emotion_folder):
            continue

        # Create emotion-specific subfolders in train and val directories
        os.makedirs(os.path.join(train_path, emotion), exist_ok=True)
        os.makedirs(os.path.join(val_path, emotion), exist_ok=True)

        # Get all image files for the current emotion and shuffle
        files = []
        for root, _, filenames in os.walk(emotion_folder):
            for filename in filenames:
                if os.path.isfile(os.path.join(root, filename)):  # Check if it's a file
                    files.append(os.path.join(root, filename))

        random.shuffle(files)

        # Split files into training and validation subsets
        train_files, val_files = train_test_split(files, test_size=0.2)

        # Copy files to their respective directories
        for f in train_files:
            shutil.copy(f, os.path.join(train_path, emotion))
        for f in val_files:
            shutil.copy(f, os.path.join(val_path, emotion))

print("Train path:", train_path)
print("Validation path:", val_path)
print("Train and validation datasets are prepared.")


In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Create an ImageDataGenerator instance
datagen = ImageDataGenerator(
    rescale=1./255,               # Normalize pixel values between 0 and 1
    rotation_range=30,            # Random rotation up to 30 degrees
    width_shift_range=0.2,        # Horizontal shift by 20% of the image width
    height_shift_range=0.2,       # Vertical shift by 20% of the image height
    shear_range=0.2,              # Shear angle up to 20%
    zoom_range=0.2,               # Random zoom within 20%
    horizontal_flip=True,         # Randomly flip images horizontally
    fill_mode='nearest'           # Fill pixels outside boundaries with nearest pixel
)

# Create training generator
train_generator = datagen.flow_from_directory(
    train_path,      # Directory containing training images
    target_size=(224, 224),       # Resize all images to 224x224
    batch_size=32,                # Batch size
    class_mode='categorical'      # Classification mode
)

val_datagen = ImageDataGenerator(rescale=1./255)  # Only rescale for validation data

val_generator = val_datagen.flow_from_directory(
    val_path,
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical'
)


In [ ]:
from tensorflow.keras.applications import ResNet101
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.layers import GlobalAveragePooling2D, BatchNormalization, Dense, Dropout
from tensorflow.keras.regularizers import l2

# Load ResNet50 with pretrained weights
base_model = ResNet101(weights='imagenet', include_top=False, input_shape=(224, 224, 3))

x = base_model.output

# Use Global Average Pooling instead of Flatten
x = GlobalAveragePooling2D()(x)

# Add a Dense layer with Batch Normalization and L2 Regularization
x = Dense(512, activation='relu', kernel_regularizer=l2(1e-4))(x)  # L2 regularization applied here
x = BatchNormalization()(x)
x = Dropout(0.5)(x)

# Add another Dense layer for feature learning with L2 Regularization
x = Dense(256, activation='relu', kernel_regularizer=l2(1e-5))(x)  # L2 regularization applied here
x = Dropout(0.5)(x)

# Output layer for classification
output = Dense(train_generator.num_classes, activation='softmax', kernel_regularizer=l2(0.01))(x)  # L2 regularization here too

# Create the model
model = Model(inputs=base_model.input, outputs=output)

# Fine-tune the last 100 layers of the base model
for layer in base_model.layers[-100:]:
    layer.trainable = True

# Compile the model
model.compile(optimizer=Adam(learning_rate=1e-4),loss='categorical_crossentropy', metrics=['accuracy'])

# Callbacks for training
early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6, verbose = 1)

In [ ]:
from sklearn.utils.class_weight import compute_class_weight

class_weights = compute_class_weight(
    'balanced',
    classes=np.unique(train_generator.classes),
    y=train_generator.classes
)
history = model.fit(
    train_generator,
    epochs=20,
    validation_data=val_generator,
    callbacks=[early_stopping, reduce_lr]
)

# Evaluate the model
val_loss, val_accuracy = model.evaluate(val_generator)
print(f"Validation Accuracy: {val_accuracy * 100:.2f}%")


In [ ]:
# Adjust the learning rate for fine-tuning
fine_tune_learning_rate = 1e-5
model.compile(
    optimizer=Adam(learning_rate=fine_tune_learning_rate),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Recalculate steps per epoch
steps_per_epoch = 37000 // 32
validation_steps = 9000// 32

# Continue fine-tuning for 5 additional epochs
history_fine_tune = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=history.epoch[-1] + 10,  # Continue for 5 more epochs
    initial_epoch=history.epoch[-1] + 1,  # Start from the last epoch
    callbacks=[early_stopping, reduce_lr]  # Retain existing callbacks
)

# Evaluate the model after fine-tuning
val_loss, val_accuracy = model.evaluate(val_generator)
print(f"Validation Accuracy after fine-tuning: {val_accuracy * 100:.2f}%")


In [ ]:
# Generate predictions for validation data
val_predictions = model.predict(val_generator)
val_pred_classes = np.argmax(val_predictions, axis=1)
val_true_classes = val_generator.classes
class_labels = list(val_generator.class_indices.keys())

# Classification report
report = classification_report(val_true_classes, val_pred_classes, target_names=class_labels)
print("Classification Report:")
print(report)

# Confusion Matrix
conf_matrix = confusion_matrix(val_true_classes, val_pred_classes)
print("Confusion Matrix:")
print(conf_matrix)

# Plot validation accuracy and loss
history_dict = history.history
if 'accuracy' in history_dict:  # Keras may log 'acc' instead
    accuracy_key = 'accuracy'
    val_accuracy_key = 'val_accuracy'
else:
    accuracy_key = 'acc'
    val_accuracy_key = 'val_acc'

# Plot accuracy
plt.figure()
plt.plot(history.history[accuracy_key], label='Training Accuracy')
plt.plot(history.history[val_accuracy_key], label='Validation Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.title('Training and Validation Accuracy')
plt.show()

# Plot loss
plt.figure()
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.title('Training and Validation Loss')
plt.show()

In [ ]:
model.save('/kaggle/working/my_model.keras')
print("Model saved to /kaggle/working/my_model")
model.save_weights('/kaggle/working/model.weights.h5')
print("Weights saved to /kaggle/working/model.weights.h5")

In [ ]:
import shutil

# Compress the entire model directory or weights file
shutil.make_archive('/kaggle/working/my_model', 'zip', '/kaggle/working/my_model')
# OR for weights
shutil.make_archive('/kaggle/working/model_weights', 'zip', '/kaggle/working/', 'model.weights.h5')

print("Model zipped for download.")
